# Spectraverse v1.0.3 data overview

Source: `data/raw/spectraverse/spectraverse-1.0.3/spectraverse-1.0.3.mgf`
(md5 `52ed27db2d5b764a636fb2ce246004aa`, 464,228 records). Processed with the
[v1.0.3 commands](../../preproc_scripts/README.md#spectraverse-v103) into
`data/proc/spectraverse_1_0_3/`; benchmark folds in `data/split/spectraverse_benchmark_1_0_3/cv0-cv9`.

Known caveats of the stored `spec_df.pkl` (see the
[Spectraverse hub](../../docs/workstreams/spectraverse.md#open-threads)):
- `inst_type == 'Other'` is the MGF `INSTRUMENT_TYPE=iontrap` block. The parser now maps it to `IT`,
  but this pickle was built before the fix; rerun step 2 to relabel it.
- `ion_mode` is correct here (`[M+Cl]-`, `[M+HCOOH-H]-`, `[M+CH3COOH-H]-` are `N`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")  # mol_df pickles hold RDKit Mol objects from a newer RDKit

DATA = Path("../../data")
SPV_DP = DATA / "proc/spectraverse_1_0_3"
SPV_OLD_DP = DATA / "proc/spectraverse_1_0_2"
SPLIT_DP = DATA / "split/spectraverse_benchmark_1_0_3"

## Load

In [ ]:
spectraverse_spec_df = pd.read_pickle(SPV_DP / "spec_df.pkl")
spectraverse_mol_df = pd.read_pickle(SPV_DP / "mol_df.pkl").drop(columns=["mol"])
spectraverse_df = spectraverse_spec_df.merge(spectraverse_mol_df, on="mol_id", how="left")
assert spectraverse_df["inchikey_s"].notnull().all()

print(f"Spectra:              {len(spectraverse_spec_df):,}")
print(f"mol_id values:        {spectraverse_spec_df['mol_id'].nunique():,}")
print(f"2D InChIKeys:         {spectraverse_df['inchikey_s'].nunique():,}")
print(f"(mol_id, adduct):     {spectraverse_spec_df.groupby(['mol_id', 'prec_type']).ngroups:,}")
spectraverse_spec_df.head()

## Metadata columns

In [ ]:
for col in ["inst_type", "ion_mode", "frag_mode", "ce_type", "col_gas"]:
    print(spectraverse_spec_df[col].value_counts(dropna=False).to_string(), "\n")

In [ ]:
# MGF instrument type by adduct ('Other' = ion trap, see caveat above)
pd.crosstab(spectraverse_spec_df["prec_type"], spectraverse_spec_df["inst_type"], margins=True)

## Adducts

In [ ]:
adduct_counts = spectraverse_spec_df["prec_type"].value_counts()
unique_mols_per_adduct = spectraverse_df.groupby("prec_type")["inchikey_s"].nunique()
adduct_summary = pd.DataFrame({
    "spectra": adduct_counts,
    "frac_spectra": (adduct_counts / adduct_counts.sum()).round(4),
    "molecules": unique_mols_per_adduct,
}).sort_values("spectra", ascending=False)
ADDUCTS = adduct_summary.index.tolist()
adduct_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in [
    (axes[0], "spectra", "Spectra per adduct"),
    (axes[1], "molecules", "Unique molecules (2D InChIKey) per adduct"),
]:
    vals = adduct_summary[col]
    vals.plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title(f"Spectraverse v1.0.3: {title}")
    ax.set_xlabel("Adduct")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)
    for i, v in enumerate(vals):
        ax.text(i, v * 1.01, f"{v:,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
spectra_per_mol_adduct = spectraverse_df.groupby(["mol_id", "prec_type"]).size()

plt.figure(figsize=(12, 5))
plt.hist(spectra_per_mol_adduct, bins=50, edgecolor="black", alpha=0.7)
plt.axvline(spectra_per_mol_adduct.mean(), color="red", linestyle="--",
            label=f"Mean: {spectra_per_mol_adduct.mean():.1f}")
plt.axvline(spectra_per_mol_adduct.median(), color="orange", linestyle="--",
            label=f"Median: {spectra_per_mol_adduct.median():.1f}")
plt.yscale("log")
plt.xlabel("Spectra per (molecule, adduct) pair")
plt.ylabel("Pairs")
plt.title("Spectra per (molecule, adduct) pair")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total (molecule, adduct) pairs: {len(spectra_per_mol_adduct):,}")
print(spectra_per_mol_adduct.describe().round(2).to_string())

## Collision energy

In [ ]:
spec_with_ace = spectraverse_spec_df[spectraverse_spec_df["ace"].notnull()]
print(f"Spectra with ACE: {len(spec_with_ace):,} "
      f"({len(spec_with_ace) / len(spectraverse_spec_df):.1%})")
print(f"Spectra with NCE: {spectraverse_spec_df['nce'].notnull().sum():,}")
print(f"(mol_id, adduct) pairs with ACE: {spec_with_ace.groupby(['mol_id', 'prec_type']).ngroups:,}")

ace_by_adduct = pd.DataFrame({
    "spectra_with_ace": spec_with_ace["prec_type"].value_counts(),
    "frac_with_ace": (spec_with_ace["prec_type"].value_counts() / adduct_counts).round(3),
    "pairs_with_ace": spec_with_ace.groupby("prec_type")["mol_id"].nunique(),
}).reindex(ADDUCTS)
ace_by_adduct

In [ ]:
# ACE coverage by instrument type
pd.DataFrame({
    "spectra": spectraverse_spec_df["inst_type"].value_counts(),
    "with_ace": spec_with_ace["inst_type"].value_counts(),
}).fillna(0).astype(int).assign(frac=lambda d: (d["with_ace"] / d["spectra"]).round(3))

In [ ]:
plt.figure(figsize=(12, 5))
plt.hist(spec_with_ace["ace"].clip(upper=150), bins=150, edgecolor="none")
plt.xlabel("ACE (eV, clipped at 150)")
plt.ylabel("Spectra")
plt.title("ACE distribution")
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## v1.0.2 vs v1.0.3

Molecules are compared by 2D InChIKey; the two builds have independent `mol_id` values.

In [ ]:
old_spec_df = pd.read_pickle(SPV_OLD_DP / "spec_df.pkl")[["mol_id", "prec_type"]]
old_mol_df = pd.read_pickle(SPV_OLD_DP / "mol_df.pkl")[["mol_id", "inchikey_s"]]
old_df = old_spec_df.merge(old_mol_df, on="mol_id", how="left")
del old_spec_df

version_cmp = pd.DataFrame({
    "spectra_1_0_2": old_df["prec_type"].value_counts(),
    "spectra_1_0_3": adduct_counts,
    "molecules_1_0_2": old_df.groupby("prec_type")["inchikey_s"].nunique(),
    "molecules_1_0_3": unique_mols_per_adduct,
}).reindex(ADDUCTS)
version_cmp.loc["total"] = [
    len(old_df), len(spectraverse_df),
    old_df["inchikey_s"].nunique(), spectraverse_df["inchikey_s"].nunique(),
]
version_cmp["spectra_change"] = version_cmp["spectra_1_0_3"] - version_cmp["spectra_1_0_2"]
version_cmp["molecule_change"] = version_cmp["molecules_1_0_3"] - version_cmp["molecules_1_0_2"]
version_cmp

In [ ]:
ik_old = set(old_df["inchikey_s"])
ik_new = set(spectraverse_df["inchikey_s"])
print(f"InChIKeys in both:          {len(ik_old & ik_new):,}")
print(f"Only in v1.0.2 (removed):   {len(ik_old - ik_new):,}")
print(f"Only in v1.0.3 (added):     {len(ik_new - ik_old):,}")
n_added_spec = spectraverse_df["inchikey_s"].isin(ik_new - ik_old).sum()
n_removed_spec = old_df["inchikey_s"].isin(ik_old - ik_new).sum()
print(f"v1.0.3 spectra of added molecules:   {n_added_spec:,}")
print(f"v1.0.2 spectra of removed molecules: {n_removed_spec:,}")

## Benchmark folds

Each cv uses the MGF `FOLD` field: one fold for test, fold 10 for val, the rest for train
(`stats.json` in each cv directory).

In [ ]:
rows = []
for cv_dp in sorted(SPLIT_DP.glob("cv*"), key=lambda p: int(p.name[2:])):
    st = json.loads((cv_dp / "stats.json").read_text())
    rows.append({
        "cv": st["cv"],
        "test_fold": st["folds"]["test"],
        "val_fold": st["folds"]["val"],
        **{f"{k}_spectra": v for k, v in st["num_spectra"].items()},
        **{f"{k}_molecules": v for k, v in st["num_molecules"].items()},
        "mgf_md5": st["mgf_md5"],
    })
fold_df = pd.DataFrame(rows).set_index("cv")
assert (fold_df["mgf_md5"] == "52ed27db2d5b764a636fb2ce246004aa").all()
fold_df.drop(columns="mgf_md5")

In [ ]:
# Adduct composition of each cv's test set (fraction of spectra)
test_comp = {}
for cv_dp in sorted(SPLIT_DP.glob("cv*"), key=lambda p: int(p.name[2:])):
    ids = pd.read_csv(cv_dp / "test_ids.csv", usecols=["spec_id"])
    sub = spectraverse_spec_df[spectraverse_spec_df["spec_id"].astype(int).isin(ids["spec_id"])]
    assert len(sub) == len(ids)
    test_comp[cv_dp.name] = sub["prec_type"].value_counts(normalize=True)
pd.DataFrame(test_comp).reindex(ADDUCTS).round(3)

## Spectral entropy and peak count

In [ ]:
import ms_entropy as me
from tqdm.auto import tqdm

tqdm.pandas(desc="Entropy")
spectraverse_spec_df["ms_entropy"] = spectraverse_spec_df["peaks"].progress_apply(
    lambda peaks: me.calculate_spectral_entropy(
        np.array(peaks), clean_spectrum=True, min_ms2_difference_in_da=0.05
    )
)
spectraverse_spec_df["num_peaks"] = spectraverse_spec_df["peaks"].apply(len)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, adduct in zip(axes.flatten(), ADDUCTS):
    vals = spectraverse_spec_df.loc[spectraverse_spec_df["prec_type"] == adduct, "ms_entropy"]
    ax.hist(vals, bins=50, alpha=0.7, edgecolor="black")
    ax.axvline(vals.mean(), color="red", linestyle="--", label=f"Mean: {vals.mean():.2f}")
    ax.set_title(f"{adduct} (n={len(vals):,})")
    ax.set_xlabel("Spectral entropy")
    ax.set_ylabel("Spectra")
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.suptitle("Spectral entropy by adduct", fontsize=16)
plt.tight_layout()
plt.show()

spectraverse_spec_df.groupby("prec_type")[["ms_entropy", "num_peaks"]].agg(
    ["mean", "std", "median"]
).reindex(ADDUCTS).round(3)

In [ ]:
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
contourf_handles = []
for ax, adduct in zip(axes.flatten(), ADDUCTS):
    sub = spectraverse_spec_df[spectraverse_spec_df["prec_type"] == adduct]
    sub = sub.sample(n=min(20_000, len(sub)), random_state=0)
    entropy = sub["ms_entropy"].values
    peak_num = sub["num_peaks"].values
    kernel = gaussian_kde(np.vstack([entropy, peak_num]))
    y_max = min(peak_num.max(), 200)
    X, Y = np.mgrid[entropy.min():entropy.max():100j, peak_num.min():y_max:100j]
    Z = kernel(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    cf = ax.contourf(X, Y, Z, levels=10, cmap="Blues", alpha=0.8)
    ax.contour(X, Y, Z, levels=10, colors="black", linewidths=0.5)
    ax.set_xlabel("Entropy")
    ax.set_ylabel("Peak count")
    ax.set_title(f"{adduct} (n={len(sub):,} sampled)")
    contourf_handles.append(cf)
fig.subplots_adjust(top=0.93, bottom=0.15, hspace=0.35)
cbar = fig.colorbar(contourf_handles[-1], ax=axes, orientation="horizontal", fraction=0.04, pad=0.06)
cbar.set_label("Density")
fig.suptitle("Spectral entropy vs. peak count by adduct", fontsize=16)
plt.show()

## Overlap with NIST HR libraries

Molecules (2D InChIKey) in Spectraverse v1.0.3 that are absent from NIST20 HR (`nist20_hr`) or
NIST23 HR (`nist23_hr1`, `nist23_hr2`). MoNA rows stored beside NIST are excluded.

In [ ]:
def nist_hr_inchikeys(proc_dp, dsets):
    spec = pd.read_pickle(proc_dp / "spec_df.pkl")[["mol_id", "dset"]]
    mol_ids = spec.loc[spec["dset"].isin(dsets), "mol_id"].unique()
    del spec
    mol = pd.read_pickle(proc_dp / "mol_df.pkl")[["mol_id", "inchikey_s"]]
    return set(mol.loc[mol["mol_id"].isin(mol_ids), "inchikey_s"])


def novel_summary(ref_inchikeys, ref_name):
    novel = spectraverse_df[~spectraverse_df["inchikey_s"].isin(ref_inchikeys)]
    novel_ace = novel[novel["ace"].notnull()]
    print(f"Spectraverse molecules not in {ref_name}: {novel['inchikey_s'].nunique():,} "
          f"of {spectraverse_df['inchikey_s'].nunique():,}")
    print(f"Spectra of those molecules: {len(novel):,} (with ACE: {len(novel_ace):,})")
    return pd.DataFrame({
        "spectra": novel["prec_type"].value_counts(),
        "molecules": novel.groupby("prec_type")["inchikey_s"].nunique(),
        "spectra_with_ace": novel_ace["prec_type"].value_counts(),
        "pairs_with_ace": novel_ace.groupby("prec_type")["mol_id"].nunique(),
    }).reindex(ADDUCTS).fillna(0).astype(int)


nist20_hr_inchikeys = nist_hr_inchikeys(DATA / "proc/nist20", ["nist20_hr"])
print(f"NIST20 HR molecules: {len(nist20_hr_inchikeys):,}")
novel_summary(nist20_hr_inchikeys, "NIST20 HR")

In [ ]:
nist23_hr_inchikeys = nist_hr_inchikeys(DATA / "proc/nist23", ["nist23_hr1", "nist23_hr2"])
print(f"NIST23 HR molecules: {len(nist23_hr_inchikeys):,}")
novel_summary(nist23_hr_inchikeys, "NIST23 HR")